In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
conn = sqlite3.connect("base_tratada.db")

query = """
SELECT *
FROM base_tratada;
"""

df = pd.read_sql(query, conn)

# Visualizar os dados
df.head()

#testando se o banco conectou

In [ ]:
df.shape
df.info()

In [ ]:
df["preco"].describe()

#antes do tratamento de outliers

In [ ]:
sns.boxplot(x=df["preco"])
plt.show()

plt.show()

#grafico para avaliar os outliers no dataframe sem tratamento dos outliers

In [ ]:
#outliers
df["preco"].sort_values(ascending=False).head(20)
#verificando valores

In [ ]:
df_tratado = df[df["preco"] <= 7000].copy()

#Limita o preco das diaria ate 13k

In [ ]:
df.shape[0], df_tratado.shape[0]

#mostra os dois dataframes

In [ ]:
print(f"Antes: {df.shape[0]} linhas")
print(f"Depois: {df_tratado.shape[0]} linhas")
print(f"Removidos: {df.shape[0] - df_tratado.shape[0]} linhas")

In [ ]:
df_tratado["preco"].describe()

In [ ]:
df["preco"].sort_values(ascending=True).head(20)

#mostra valores em forma crescente

In [ ]:
df_tratado_price_min = df_tratado[
    (df_tratado["preco"] >= 65) &
    (df_tratado["quartos"] <= 15) &
    (df_tratado["banheiros"] <= 12)
].copy()

#atribui ao novo dataframe valores de limete pós remocao de outliers

In [ ]:
print(df_tratado_price_min["preco"].quantile([0.90, 0.95, 0.99, 1.0]))

In [ ]:
print(f"Antes: {df_tratado.shape[0]} linhas")
print(f"Depois: {df_tratado_price_min.shape[0]} linhas")
print(f"Removidos: {df_tratado.shape[0] - df_tratado_price_min.shape[0]} linhas")

In [ ]:
df_tratado_price_min["preco"].describe()

In [ ]:
sns.boxplot(x="hospedes", y="preco", data=df_tratado_price_min)
plt.show()

In [ ]:
sns.boxplot(x="hospedes", y="preco", data=df_tratado_price_min, showfliers=False)
plt.show()

In [ ]:
print("Registros originais:", len(df_tratado))
print("Registros após remoção de outliers:", len(df_tratado_price_min))

In [ ]:
plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df_tratado_price_min,
    x="quartos",
    y="preco"
)

plt.title("Distribuição do preço por número de quartos")
plt.xlim(-0.5, 11.5)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df_tratado_price_min,
    x="quartos",
    y="preco",
    showfliers=False
)

plt.title("Distribuição do preço por número de quartos")
plt.xlim(-0.5, 11.5)
plt.show()

In [ ]:

df_box_banheiros = df_tratado_price_min[df_tratado_price_min["banheiros"] <=10].copy()

In [ ]:
plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df_box_banheiros,
    x="banheiros",
    y="preco"
)

plt.title("Distribuição do preço por número de banheiros (até 10)")
plt.xlabel("Número de banheiros")
plt.ylabel("Preço da diária")

plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df_box_banheiros,
    x="banheiros",
    y="preco",
    showfliers=False
)

plt.title("Distribuição do preço por número de banheiros (até 10)")
plt.xlabel("Número de banheiros")
plt.ylabel("Preço da diária")

plt.show()

In [ ]:
mapa_tipos = {
    "tipo_quarto_Entire home/apt": "Casa/Apto Inteiro",
    "tipo_quarto_Hotel room": "Quarto de Hotel",
    "tipo_quarto_Private room": "Quarto Privado",
    "tipo_quarto_Shared room": "Quarto Compartilhado",
    "tipo_de_propriedade_grupo": "Outros"
}

df_boxplot = df_tratado_price_min.copy()

df_boxplot["Tipo de hospedagem"] = None

for coluna, nome in mapa_tipos.items():
    df_boxplot.loc[df_boxplot[coluna] == 1, "Tipo de hospedagem"] = nome


ordem = (
    df_boxplot
    .groupby("Tipo de hospedagem")["preco"]
    .median()
    .sort_values(ascending=False)
    .index
)

plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df_boxplot,
    x="Tipo de hospedagem",
    y="preco",
    order=ordem,
    hue="Tipo de hospedagem",
    palette="viridis",
    legend=False
)

plt.title("Distribuição do preço por tipo de hospedagem")
plt.xlabel("Tipo de hospedagem")
plt.ylabel("Preço da diária")
plt.xticks(rotation=30)
plt.show()

In [ ]:
ordem = (
    df_boxplot
    .groupby("Tipo de hospedagem")["preco"]
    .median()
    .sort_values(ascending=False)
    .index
)

plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df_boxplot,
    x="Tipo de hospedagem",
    y="preco",
    order=ordem,
    hue="Tipo de hospedagem",
    palette="viridis",
    legend=False,
    showfliers=False
)

plt.title("Distribuição do preço por tipo de hospedagem")
plt.xlabel("Tipo de hospedagem")
plt.ylabel("Preço da diária")
plt.xticks(rotation=30)
plt.show()

In [ ]:
df_tratado_price_min[
    [
        "preco",
        "hospedes",
        "quartos",
        "camas",
        "nota_avaliacao",
        "noites_minimas",
        "quantidade_avaliacoes",
        "latitude",
        "longitude",
        "banheiros",
        "tipo_quarto_Entire home/apt",
        "tipo_quarto_Hotel room",
        "tipo_quarto_Private room",
        "tipo_quarto_Shared room"
    ]
].describe()

In [ ]:
plt.figure(figsize=(10, 8))
sns.scatterplot(
    data=df_tratado_price_min,
    x="longitude",
    y="latitude",
    hue="preco",
    palette="viridis",
    alpha=0.6
)

plt.title("Distribuição geográfica dos preços")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.legend(title="Preço", loc="best")
plt.show()

In [ ]:
df_tratado_price_min["lat_bin"] = df_tratado_price_min["latitude"].round(2)
df_tratado_price_min["lon_bin"] = df_tratado_price_min["longitude"].round(2)

In [ ]:
media_localizacao = (
    df_tratado_price_min
    .groupby(["lat_bin", "lon_bin"])["preco"]
    .mean()
    .reset_index()
)

In [ ]:
plt.figure(figsize=(10, 8))
sns.scatterplot(
    data=media_localizacao,
    x="lon_bin",
    y="lat_bin",
    size="preco",
    hue="preco",
    palette="coolwarm",
    sizes=(20, 400),
    alpha=0.8
)

plt.title("Preço médio por localidade")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()

In [ ]:
tipos_imovel = {
    "Casa/apt Inteiro": "tipo_quarto_Entire home/apt",
    "Quarto de Hotel": "tipo_quarto_Hotel room",
    "Quarto Privado": "tipo_quarto_Private room",
    "Quarto Compartilhado": "tipo_quarto_Shared room"
}

In [ ]:
frequencias = []
total_registros = len(df_tratado_price_min)

for nome, coluna in tipos_imovel.items():
    freq_absoluta = df_tratado_price_min[coluna].sum()
    freq_relativa = freq_absoluta / total_registros

    frequencias.append({
        "Tipo de imóvel": nome,
        "Frequência relativa": freq_relativa
    })

df_frequencia = pd.DataFrame(frequencias)
df_frequencia["Frequência (%)"] = df_frequencia["Frequência relativa"] * 100

# =========================
# TABELA FINAL
# =========================

df_frequencia

In [ ]:
tipos_imovel = {
    "Casa/apt Inteiro": "tipo_quarto_Entire home/apt",
    "Quarto de Hotel": "tipo_quarto_Hotel room",
    "Quarto Privado": "tipo_quarto_Private room",
    "Quarto Compartilhado": "tipo_quarto_Shared room",
    "Outros": "tipo_de_propriedade_grupo"
}

In [ ]:
media_precos = []

for nome, coluna in tipos_imovel.items():
    preco_medio = df_tratado_price_min.loc[
        df_tratado_price_min[coluna] == 1, "preco"
    ].mean()
    
    media_precos.append({
        "Tipo de imóvel": nome,
        "Preço médio": preco_medio
    })

df_media = pd.DataFrame(media_precos)




df_media

In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    data=df_media,
    x="Tipo de imóvel",
    y="Preço médio",
    palette="viridis"
)

df_media = df_media.sort_values(
    by="Preço médio",
    ascending=False
)

plt.title("Preço médio da diária por tipo de imóvel")
plt.xlabel("Tipo de imóvel")
plt.ylabel("Preço médio da diária")
plt.xticks(rotation=30)
plt.show()

In [ ]:
tipos_imovel = {
    "Casa/apt Inteiro": "tipo_quarto_Entire home/apt",
    "Quarto de Hotel": "tipo_quarto_Hotel room",
    "Quarto Privado": "tipo_quarto_Private room",
    "Quarto Compartilhado": "tipo_quarto_Shared room",
    "Outros": "tipo_de_propriedade_grupo"
}



mediana_precos = []

for nome, coluna in tipos_imovel.items():
    preco_mediano = df_tratado_price_min.loc[
        df_tratado_price_min[coluna] == 1, "preco"
    ].median()
    
    mediana_precos.append({
        "Tipo de imóvel": nome,
        "Preço mediano": preco_mediano
    })

df_mediana = pd.DataFrame(mediana_precos)

df_mediana


df_mediana = df_mediana.sort_values(
    by="Preço mediano",
    ascending=False
)


plt.figure(figsize=(10, 6))

sns.barplot(
    data=df_mediana,
    x="Tipo de imóvel",
    y="Preço mediano",
    palette="viridis"
)

plt.title("Preço mediano da diária por tipo de imóvel")
plt.xlabel("Tipo de imóvel")
plt.ylabel("Preço mediano da diária")
plt.xticks(rotation=30)
plt.show()

In [ ]:
df_tratado_price_min[df_tratado_price_min["bairro"] == "Copacabana"]["preco"].mean()

In [ ]:
top_bairros = (
    df_tratado_price_min
    .groupby("bairro")
    .size()
    .reset_index(name="total_anuncios")
    .sort_values("total_anuncios", ascending=False)
    .head(10)
)

In [ ]:
df_media_acomodacao = (
    df_tratado_price_min
    .groupby("bairro")["preco"]
    .mean()
    .reset_index()
)

# manter apenas bairros mais procurados
df_media_acomodacao = df_media_acomodacao.merge(
    top_bairros[["bairro"]],
    on="bairro",
    how="inner"
)

In [ ]:
df_media_acomodacao = df_media_acomodacao.sort_values("preco", ascending=False)

In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    data=df_media_acomodacao,
    x="bairro",
    y="preco",
    palette="viridis"
)

plt.title("Valor médio da diária nos bairros mais procurados por turistas")
plt.xlabel("Bairro")
plt.ylabel("Valor médio da diária (R$)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
top_bairros = (
    df_tratado_price_min
    .groupby("bairro")
    .size()
    .reset_index(name="total_anuncios")
    .sort_values("total_anuncios", ascending=False)
    .head(10)
)


df_mediana_acomodacao = (
    df_tratado_price_min
    .groupby("bairro")["preco"]
    .median()
    .reset_index()
)


df_mediana_acomodacao = df_mediana_acomodacao.merge(
    top_bairros[["bairro"]],
    on="bairro",
    how="inner"
)


df_mediana_acomodacao = df_mediana_acomodacao.sort_values(
    "preco",
    ascending=False
)


plt.figure(figsize=(10, 6))

sns.barplot(
    data=df_mediana_acomodacao,
    x="bairro",
    y="preco",
    palette="viridis"
)

plt.title("Valor mediano da diária nos bairros mais procurados por turistas")
plt.xlabel("Bairro")
plt.ylabel("Valor mediano da diária (R$)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
df_top_bairros = df_tratado_price_min.merge(
    top_bairros[["bairro"]],
    on="bairro",
    how="inner"
)


ordem_bairros = (
    df_top_bairros
    .groupby("bairro")["preco"]
    .median()
    .sort_values()
    .index
)


plt.figure(figsize=(12, 6))

sns.boxplot(
    data=df_top_bairros,
    x="bairro",
    y="preco",
    order=ordem_bairros,
    palette="viridis"
)

plt.title("Distribuição do preço da diária nos bairros mais procurados")
plt.xlabel("Bairro")
plt.ylabel("Preço da diária (R$)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))

sns.boxplot(
    data=df_top_bairros,
    x="bairro",
    y="preco",
    order=ordem_bairros,
    palette="viridis",
    showfliers=False
)

plt.title("Distribuição do preço da diária nos bairros mais procurados")
plt.xlabel("Bairro")
plt.ylabel("Preço da diária (R$)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
variaveis = [
    "preco",
    "quartos",
    "banheiros",
    "latitude",
    "longitude",
    "camas",
    "nota_avaliacao",
    "quantidade_avaliacoes",
    "tipo_quarto_Entire home/apt",
    "tipo_quarto_Hotel room",
    "tipo_quarto_Private room",
    "tipo_quarto_Shared room"
]

df_corr = df_tratado_price_min[variaveis]

In [ ]:
corr_matrix = df_corr.corr()

In [ ]:
plt.figure(figsize=(12, 9))

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    linewidths=0.5
)

plt.title("Matriz de Correlação das Variáveis com o Preço da Diária")
plt.show()

AQUI COMEÇAM AS COISAS DO MODELO - LINEAR REGRESSION

In [ ]:
y = df_tratado_price_min["preco"]

X = df_tratado_price_min[[
    "latitude", 
    "longitude",
    "tipo_quarto_Entire home/apt",
    "quartos",
    "banheiros"
]]

In [ ]:
from sklearn.model_selection import train_test_split

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
from sklearn.linear_model import LinearRegression

modelo = LinearRegression()
modelo.fit(X_treino, y_treino)

In [ ]:
y_pred = modelo.predict(X_teste)

In [ ]:
y_pred

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

print("MSE:", mean_squared_error(y_teste, y_pred))
print("R²:", r2_score(y_teste, y_pred))

In [ ]:
importancia = pd.DataFrame({
    "Variável": X.columns,
    "Coeficiente": modelo.coef_
}).sort_values(by="Coeficiente", key=abs, ascending=False)

importancia

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(y_teste, y_pred)
plt.xlabel("Valor real")
plt.ylabel("Valor previsto")
plt.show()